In [ ]:
# SPARC FROZEN VALIDATION REPRODUCTION — CELL 1
# Setup, package lock, input integrity check, and locked-data loading.

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import subprocess
import sys

RELEASE_ROOT = Path(
    "/content/drive/MyDrive/UQSH_DarkMatter_Project/"
    "reproducibility_exports/"
    "sparc-clean-independent-reconstruction-20260812_124457_UTC/"
    "SPARC_Clean_Independent_Reconstruction"
)

LOCK_FILE = RELEASE_ROOT / "requirements-lock.txt"
MANIFEST_FILE = RELEASE_ROOT / "stage1_manifest.json"

if not RELEASE_ROOT.is_dir():
    raise FileNotFoundError(f"Release folder not found:\n{RELEASE_ROOT}")

if not LOCK_FILE.is_file():
    raise FileNotFoundError(f"Package lock not found:\n{LOCK_FILE}")

if not MANIFEST_FILE.is_file():
    raise FileNotFoundError(f"Manifest not found:\n{MANIFEST_FILE}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(LOCK_FILE)],
    check=True,
)

import numpy as np
import pandas as pd

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

manifest = json.loads(MANIFEST_FILE.read_text(encoding="utf-8"))
integrity_problems = []

for entry in manifest["copied_files"]:
    relative_path = entry["release_relative_path"]
    expected_hash = entry["copied_sha256"]
    path = RELEASE_ROOT / relative_path

    if not path.is_file():
        integrity_problems.append(f"MISSING: {relative_path}")
    elif sha256_file(path) != expected_hash:
        integrity_problems.append(f"HASH MISMATCH: {relative_path}")

if integrity_problems:
    print("=== INPUT INTEGRITY: FAIL ===")

    for problem in integrity_problems:
        print("-", problem)

    raise RuntimeError("The locked reproduction inputs are not intact.")

INPUT_DIR = RELEASE_ROOT / "inputs"
REFERENCE_DIR = RELEASE_ROOT / "locked_reference" / "kernel_variant_nested_validation"

MASTER_FILE = (
    REFERENCE_DIR / "tables" / "kernel_variant_master_table.csv"
)

FOLD_RESULTS_FILE = (
    REFERENCE_DIR / "tables" / "kernel_variant_nested_fold_results.csv"
)

PERFORMANCE_FILE = (
    REFERENCE_DIR / "tables" / "kernel_variant_performance_summary.csv"
)

required_files = [
    MASTER_FILE,
    FOLD_RESULTS_FILE,
    PERFORMANCE_FILE,
]

for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(f"Required locked file missing:\n{path}")

master = pd.read_csv(MASTER_FILE, low_memory=False)
locked_folds = pd.read_csv(FOLD_RESULTS_FILE, low_memory=False)
locked_performance = pd.read_csv(PERFORMANCE_FILE, low_memory=False)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
RUN_DIR = RELEASE_ROOT / "public_colab_runs" / f"frozen_validation_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("=== CELL 1 COMPLETE ===")
print("Input integrity: PASS")
print("Master table:", master.shape)
print("Frozen fold table:", locked_folds.shape)
print("Performance reference:", locked_performance.shape)
print("New output folder:", RUN_DIR)

# SPARC FROZEN VALIDATION REPRODUCTION — CELL 2
# Exact re-execution of all 60 frozen outer-fold model fits.

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

TARGET_COLUMN = "log_g_field_obs"

COMMON_FEATURES = [
    "log_r_over_Rdisk",
    "log_local_SBdisk",
]

required_columns = [
    "Galaxy",
    "sample_analysis",
    "outer_fold",
    TARGET_COLUMN,
] + COMMON_FEATURES

missing_columns = [
    column for column in required_columns
    if column not in master.columns
]

if missing_columns:
    raise KeyError(
        "Required master-table columns missing:\n"
        + "\n".join(missing_columns)
    )

frozen_folds = locked_folds[
    locked_folds["selection_mode"] == "frozen_reference"
].copy()

if len(frozen_folds) != 60:
    raise RuntimeError(
        f"Expected 60 frozen fold specifications, found {len(frozen_folds)}."
    )

def parse_global_features(value):
    if pd.isna(value):
        return []

    text = str(value).strip()

    if text == "" or text.lower() in {"none", "nan"}:
        return []

    return [
        item.strip()
        for item in text.split(",")
        if item.strip()
    ]

oof_tables = []
fold_check_rows = []

for _, setting in frozen_folds.iterrows():
    sample_name = str(setting["sample"])
    kernel_variant = str(setting["kernel_variant"])
    outer_fold = int(setting["outer_fold"])
    kernel_feature = str(setting["selected_kernel_feature"])
    global_features = parse_global_features(
        setting["selected_global_features"]
    )
    ridge_alpha = float(setting["selected_ridge_alpha"])

    feature_columns = (
        [kernel_feature] +
        COMMON_FEATURES +
        global_features
    )

    missing_features = [
        column for column in feature_columns
        if column not in master.columns
    ]

    if missing_features:
        raise KeyError(
            f"Features missing for {sample_name}, {kernel_variant}, "
            f"fold {outer_fold}:\n"
            + "\n".join(missing_features)
        )

    use = master[
        master["sample_analysis"].astype(str) == sample_name
    ].copy()

    use = use[
        [
            "Galaxy",
            "sample_analysis",
            "outer_fold",
            "g_bar_m_s2",
            "log_g_obs",
            TARGET_COLUMN,
        ] + feature_columns
    ].replace([np.inf, -np.inf], np.nan).dropna().copy()

    train = use[use["outer_fold"] != outer_fold].copy()
    test = use[use["outer_fold"] == outer_fold].copy()

    if len(train) != int(setting["n_train_rows"]):
        raise RuntimeError(
            f"Training-row mismatch for {sample_name}, {kernel_variant}, "
            f"fold {outer_fold}: {len(train)} vs "
            f"{int(setting['n_train_rows'])}"
        )

    if len(test) != int(setting["n_test_rows"]):
        raise RuntimeError(
            f"Test-row mismatch for {sample_name}, {kernel_variant}, "
            f"fold {outer_fold}: {len(test)} vs "
            f"{int(setting['n_test_rows'])}"
        )

    model = Pipeline([
        ("scale", StandardScaler()),
        ("ridge", Ridge(alpha=ridge_alpha)),
    ])

    model.fit(
        train[feature_columns].to_numpy(),
        train[TARGET_COLUMN].to_numpy(),
    )

    prediction = model.predict(
        test[feature_columns].to_numpy()
    )

    output = test[
        [
            "Galaxy",
            "sample_analysis",
            "outer_fold",
            "g_bar_m_s2",
            "log_g_obs",
            TARGET_COLUMN,
        ]
    ].copy()

    output = output.rename(
        columns={"sample_analysis": "sample"}
    )

    output["kernel_variant"] = kernel_variant
    output["selected_kernel_feature"] = kernel_feature
    output["selected_global_features"] = ",".join(global_features)
    output["selected_ridge_alpha"] = ridge_alpha
    output["prediction_frozen_design_oof"] = prediction
    output["residual_frozen_design_oof"] = (
        test[TARGET_COLUMN].to_numpy() - prediction
    )

    oof_tables.append(output)

    fold_check_rows.append({
        "sample": sample_name,
        "kernel_variant": kernel_variant,
        "outer_fold": outer_fold,
        "selected_kernel_feature": kernel_feature,
        "selected_global_features": ",".join(global_features),
        "selected_ridge_alpha": ridge_alpha,
        "n_train_rows": len(train),
        "n_test_rows": len(test),
        "reconstructed_test_rmse": float(
            mean_squared_error(
                test[TARGET_COLUMN].to_numpy(),
                prediction,
            ) ** 0.5
        ),
    })

exact_oof = pd.concat(oof_tables, ignore_index=True)
exact_fold_check = pd.DataFrame(fold_check_rows)

field_summary_rows = []

for (sample_name, kernel_variant), group in exact_oof.groupby(
    ["sample", "kernel_variant"],
    sort=True,
):
    y_true = group[TARGET_COLUMN].to_numpy()
    y_pred = group["prediction_frozen_design_oof"].to_numpy()

    field_summary_rows.append({
        "sample": sample_name,
        "kernel_variant": kernel_variant,
        "n_rows": int(len(group)),
        "n_galaxies": int(group["Galaxy"].nunique()),
        "reconstructed_field_rmse": float(
            mean_squared_error(y_true, y_pred) ** 0.5
        ),
        "reconstructed_field_r2": float(
            r2_score(y_true, y_pred)
        ),
    })

field_summary = pd.DataFrame(field_summary_rows)

locked_field = locked_performance[
    locked_performance["selection_mode"] == "frozen_reference"
][
    [
        "sample",
        "kernel_variant",
        "n_rows",
        "n_galaxies",
        "field_rmse",
        "field_r2",
    ]
].copy()

field_comparison = field_summary.merge(
    locked_field,
    on=["sample", "kernel_variant"],
    how="left",
    suffixes=("_reconstructed", "_locked"),
)

field_comparison["rmse_difference"] = (
    field_comparison["reconstructed_field_rmse"] -
    field_comparison["field_rmse"]
)

field_comparison["r2_difference"] = (
    field_comparison["reconstructed_field_r2"] -
    field_comparison["field_r2"]
)

field_comparison["field_metrics_exact"] = (
    field_comparison["rmse_difference"].abs() < 1e-10
) & (
    field_comparison["r2_difference"].abs() < 1e-10
)

exact_oof.to_csv(
    RUN_DIR / "frozen_design_oof_predictions.csv",
    index=False,
)

exact_fold_check.to_csv(
    RUN_DIR / "frozen_design_fold_check.csv",
    index=False,
)

field_comparison.to_csv(
    RUN_DIR / "frozen_design_field_vs_locked_comparison.csv",
    index=False,
)

all_field_exact = bool(
    field_comparison["field_metrics_exact"].all()
)

print("=== CELL 2 COMPLETE ===")
print("Reconstructed OOF rows:", len(exact_oof))
print("Reconstructed fold fits:", len(exact_fold_check))
print("All field metrics exact:", all_field_exact)
print()

display(
    field_comparison.sort_values(
        ["sample", "kernel_variant"]
    ).reset_index(drop=True)
)

# SPARC FROZEN VALIDATION REPRODUCTION — CELL 3
# Exact reconstruction check for the derived observed acceleration g_obs.

from sklearn.metrics import mean_squared_error, r2_score

if "exact_oof" not in globals():
    raise RuntimeError("Run CELL 2 first.")

required_columns = [
    "g_bar_m_s2",
    "log_g_obs",
    "prediction_frozen_design_oof",
]

missing_columns = [
    column for column in required_columns
    if column not in exact_oof.columns
]

if missing_columns:
    raise KeyError(
        "Required OOF columns missing:\n"
        + "\n".join(missing_columns)
    )

gobs_check = exact_oof.copy()

gobs_check["g_field_pred_m_s2"] = np.power(
    10.0,
    gobs_check["prediction_frozen_design_oof"]
)

gobs_check["g_obs_pred_m_s2"] = (
    gobs_check["g_bar_m_s2"] +
    gobs_check["g_field_pred_m_s2"]
)

gobs_check["log_g_obs_pred"] = np.log10(
    gobs_check["g_obs_pred_m_s2"]
)

gobs_summary_rows = []

for (sample_name, kernel_variant), group in gobs_check.groupby(
    ["sample", "kernel_variant"],
    sort=True,
):
    y_true = group["log_g_obs"].to_numpy()
    y_pred = group["log_g_obs_pred"].to_numpy()

    gobs_summary_rows.append({
        "sample": sample_name,
        "kernel_variant": kernel_variant,
        "n_rows": int(len(group)),
        "reconstructed_gobs_rmse": float(
            mean_squared_error(y_true, y_pred) ** 0.5
        ),
        "reconstructed_gobs_r2": float(
            r2_score(y_true, y_pred)
        ),
    })

gobs_summary = pd.DataFrame(gobs_summary_rows)

locked_gobs = locked_performance[
    locked_performance["selection_mode"] == "frozen_reference"
][
    [
        "sample",
        "kernel_variant",
        "gobs_rmse",
        "gobs_r2",
    ]
].copy()

gobs_comparison = gobs_summary.merge(
    locked_gobs,
    on=["sample", "kernel_variant"],
    how="left",
)

gobs_comparison["gobs_rmse_difference"] = (
    gobs_comparison["reconstructed_gobs_rmse"] -
    gobs_comparison["gobs_rmse"]
)

gobs_comparison["gobs_r2_difference"] = (
    gobs_comparison["reconstructed_gobs_r2"] -
    gobs_comparison["gobs_r2"]
)

gobs_comparison["gobs_metrics_exact"] = (
    gobs_comparison["gobs_rmse_difference"].abs() < 1e-10
) & (
    gobs_comparison["gobs_r2_difference"].abs() < 1e-10
)

gobs_check.to_csv(
    RUN_DIR / "frozen_design_gobs_oof_predictions.csv",
    index=False,
)

gobs_comparison.to_csv(
    RUN_DIR / "frozen_design_gobs_vs_locked_comparison.csv",
    index=False,
)

all_gobs_exact = bool(
    gobs_comparison["gobs_metrics_exact"].all()
)

print("=== CELL 3 COMPLETE ===")
print("Reconstructed rows:", len(gobs_check))
print("All g_obs metrics exact:", all_gobs_exact)
print()

display(
    gobs_comparison.sort_values(
        ["sample", "kernel_variant"]
    ).reset_index(drop=True)
)



# ============================================================
# FIGURE PREPARATION — certified source tables for Cells 15 and 16
# The figures are derived from the just-reconstructed OOF table.
# No frozen release file is modified; all artefacts stay in RUN_DIR.
# ============================================================

FIGURE_DIR = RUN_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
(FIGURE_DIR / "reader_oriented_exploration").mkdir(parents=True, exist_ok=True)

q12 = gobs_check[gobs_check["sample"].astype(str).eq("Q12")].copy()
if q12.empty:
    raise RuntimeError("The reconstructed OOF table does not contain the Q12 sample.")

q12["log_g_bar"] = np.log10(q12["g_bar_m_s2"].to_numpy(dtype=float))
q12["observed_log_g_obs"] = q12["log_g_obs"].to_numpy(dtype=float)
q12["predicted_log_g_obs"] = q12["log_g_obs_pred"].to_numpy(dtype=float)
q12["residual_dex"] = q12["predicted_log_g_obs"] - q12["observed_log_g_obs"]

if not np.allclose(
    q12["residual_dex"].to_numpy(),
    q12["log_g_obs_pred"].to_numpy() - q12["log_g_obs"].to_numpy(),
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError("Derived total-acceleration residual audit failed.")

EXPLORATION_SOURCE = (
    FIGURE_DIR / "reader_oriented_exploration" / "q12_oof_reader_oriented_source_data.csv"
)
q12[
    [
        "Galaxy", "sample", "outer_fold", "kernel_variant", "log_g_bar",
        "observed_log_g_obs", "predicted_log_g_obs", "residual_dex",
    ]
].to_csv(EXPLORATION_SOURCE, index=False)

# Construct the rotation-curve table at one row per radial point from the
# four certified Q12 OOF predictions, then attach frozen radial observables.
radial_keys = ["Galaxy", "outer_fold", "g_bar_m_s2", "log_g_obs"]
q12_curve = (
    q12.groupby(radial_keys, as_index=False, sort=True)
    .agg(
        V_oof_log_median=("log_g_obs_pred", "median"),
        V_oof_log_min=("log_g_obs_pred", "min"),
        V_oof_log_max=("log_g_obs_pred", "max"),
        n_variants=("kernel_variant", "nunique"),
    )
)
if not q12_curve["n_variants"].eq(4).all():
    raise RuntimeError("Every Figure 1 Q12 radial point must have four frozen variants.")

radial_columns = ["Galaxy", "outer_fold", "g_bar_m_s2", "log_g_obs", "r_kpc", "Vobs_kms", "e_Vobs_kms", "Vbar_kms"]
if set(radial_columns).issubset(master.columns):
    radial_source = master.loc[
        master["sample_analysis"].astype(str).eq("Q12"), radial_columns
    ].copy()
else:
    candidate_files = [
        INPUT_DIR / "sparc_field_excess_observed.csv",
        INPUT_DIR / "sparc_radial_observables_with_rar_baseline.csv",
    ]
    input_path = next((p for p in candidate_files if p.is_file()), None)
    if input_path is None:
        raise FileNotFoundError("No frozen radial-observable table was found for Figure 1.")
    radial_source = pd.read_csv(input_path, low_memory=False)
    required_radial = ["Galaxy", "g_bar_m_s2", "log_g_obs", "r_kpc", "Vobs_kms", "e_Vobs_kms", "Vbar_kms"]
    missing_radial = [c for c in required_radial if c not in radial_source.columns]
    if missing_radial:
        raise KeyError(f"Frozen radial-observable table lacks: {missing_radial}")
    radial_source = radial_source[required_radial].copy()
    q12_curve = q12_curve.drop(columns="outer_fold")
    radial_keys = ["Galaxy", "g_bar_m_s2", "log_g_obs"]

for column in ["g_bar_m_s2", "log_g_obs"]:
    q12_curve[column] = pd.to_numeric(q12_curve[column], errors="raise").round(12)
    radial_source[column] = pd.to_numeric(radial_source[column], errors="raise").round(12)

radial_source = radial_source.drop_duplicates(radial_keys, keep=False)
curve_data = q12_curve.merge(radial_source, on=radial_keys, how="left", validate="one_to_one")
if curve_data[["r_kpc", "Vobs_kms", "Vbar_kms"]].isna().any().any():
    raise RuntimeError("One or more Q12 radial OOF records could not be matched to frozen observables.")

KPC_TO_M = 3.085677581491367e19
for label, log_column in [("median", "V_oof_log_median"), ("min", "V_oof_log_min"), ("max", "V_oof_log_max")]:
    curve_data[f"V_oof_{label}_kms"] = (
        np.sqrt(np.power(10.0, curve_data[log_column].to_numpy(dtype=float))
                * curve_data["r_kpc"].to_numpy(dtype=float) * KPC_TO_M) / 1000.0
    )

curve_data = curve_data[
    ["Galaxy", "r_kpc", "Vobs_kms", "e_Vobs_kms", "Vbar_kms",
     "V_oof_median_kms", "V_oof_min_kms", "V_oof_max_kms", "n_variants"]
].sort_values(["Galaxy", "r_kpc"], kind="stable")

selection_definition = pd.DataFrame({
    "Galaxy": ["NGC0247", "NGC3953", "UGC08286", "NGC2915"],
    "selection_percentile": [20, 40, 60, 80],
})
selection_metrics = (
    curve_data.assign(squared_error=(curve_data["V_oof_median_kms"] - curve_data["Vobs_kms"]) ** 2)
    .groupby("Galaxy", as_index=False)
    .agg(
        galaxy_rmse_kms=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        n_radial_rows=("r_kpc", "size"),
    )
)
selected = selection_definition.merge(selection_metrics, on="Galaxy", how="left", validate="one_to_one")
if selected[["galaxy_rmse_kms", "n_radial_rows"]].isna().any().any():
    raise RuntimeError("A fixed representative galaxy is absent from the certified Q12 reconstruction.")

SOURCE_TABLE_H = FIGURE_DIR / "figure_candidate_h_representative_galaxy_curves_source_data.csv"
SELECTION_TABLE_H = FIGURE_DIR / "figure_candidate_h_representative_galaxy_selection.csv"
curve_data[curve_data["Galaxy"].isin(selected["Galaxy"])].to_csv(SOURCE_TABLE_H, index=False)
selected.to_csv(SELECTION_TABLE_H, index=False)

print("=== FIGURE SOURCE PREPARATION COMPLETE ===")
print("Q12 OOF records:", len(q12))
print("Q12 unique radial rows:", len(curve_data))
print("Figure 1 source:", SOURCE_TABLE_H)
print("Figure 2 source:", EXPLORATION_SOURCE)




# ============================================================
# CELL 15 — FIGURE CANDIDATE H (COMPACT PANEL LABELS)
# ============================================================

import matplotlib.pyplot as plt
from IPython.display import display, Image

SOURCE_TABLE = FIGURE_DIR / "figure_candidate_h_representative_galaxy_curves_source_data.csv"
SELECTION_TABLE = FIGURE_DIR / "figure_candidate_h_representative_galaxy_selection.csv"
curve_data = pd.read_csv(SOURCE_TABLE)
selected = pd.read_csv(SELECTION_TABLE)

required_curve_columns = {"Galaxy", "r_kpc", "Vobs_kms", "e_Vobs_kms", "Vbar_kms", "V_oof_median_kms", "V_oof_min_kms", "V_oof_max_kms", "n_variants"}
required_selection_columns = {"Galaxy", "selection_percentile", "galaxy_rmse_kms", "n_radial_rows"}
missing_curve = sorted(required_curve_columns - set(curve_data.columns))
missing_selection = sorted(required_selection_columns - set(selected.columns))
if missing_curve or missing_selection:
    raise RuntimeError(f"Missing columns — curve data: {missing_curve}; selection table: {missing_selection}")
if len(selected) != 4 or not curve_data["n_variants"].eq(4).all():
    raise RuntimeError("Figure 1 needs four selected galaxies and four frozen variants per radius.")

percentile_labels = {20: "Lower-error representative", 40: "Typical-error representative", 60: "Intermediate-error representative", 80: "Higher-error representative"}
selected = selected.sort_values("selection_percentile", kind="stable").reset_index(drop=True)
fig, axes = plt.subplots(2, 2, figsize=(13.3, 10.4), constrained_layout=True)
axes = axes.ravel()

for ax, (_, info) in zip(axes, selected.iterrows()):
    galaxy = str(info["Galaxy"])
    percentile = int(info["selection_percentile"])
    curve = curve_data.loc[curve_data["Galaxy"].astype(str).eq(galaxy)].sort_values("r_kpc").copy()
    ax.fill_between(curve["r_kpc"], curve["V_oof_min_kms"], curve["V_oof_max_kms"], color="#4C9ED9", alpha=0.32, label="OOF spread across four frozen variants", zorder=1)
    ax.plot(curve["r_kpc"], curve["V_oof_min_kms"], color="#4C9ED9", linewidth=0.95, alpha=0.95, zorder=2)
    ax.plot(curve["r_kpc"], curve["V_oof_max_kms"], color="#4C9ED9", linewidth=0.95, alpha=0.95, zorder=2)
    ax.plot(curve["r_kpc"], curve["V_oof_median_kms"], color="#146EB4", linewidth=2.7, marker="o", markersize=4.1, label="Median OOF reconstruction", zorder=4)
    ax.plot(curve["r_kpc"], curve["Vbar_kms"], color="#D89C23", linewidth=2.2, linestyle="--", label="Baryonic rotation curve", zorder=3)
    valid_error = np.isfinite(curve["e_Vobs_kms"]) & (curve["e_Vobs_kms"] > 0)
    ax.errorbar(curve.loc[valid_error, "r_kpc"], curve.loc[valid_error, "Vobs_kms"], yerr=curve.loc[valid_error, "e_Vobs_kms"], fmt="o", markersize=4.0, color="#171717", ecolor="#777777", elinewidth=0.9, capsize=1.8, label="Observed rotation curve", zorder=5)
    if (~valid_error).any():
        ax.scatter(curve.loc[~valid_error, "r_kpc"], curve.loc[~valid_error, "Vobs_kms"], s=22, color="#171717", label="Observed rotation curve", zorder=5)
    median_curve_rmse = float(np.sqrt(np.mean(np.square(curve["V_oof_median_kms"].to_numpy(dtype=float) - curve["Vobs_kms"].to_numpy(dtype=float)))))
    max_variant_spread = float(np.max(curve["V_oof_max_kms"].to_numpy(dtype=float) - curve["V_oof_min_kms"].to_numpy(dtype=float)))
    ax.set_title(f"{galaxy} · {percentile_labels[percentile]} ({percentile}th percentile)", fontsize=12.5, fontweight="bold", pad=27)
    ax.text(0.5, 1.015, f"{len(curve)} radial rows  ·  RMSE {median_curve_rmse:.1f} km s$^{{-1}}$  ·  maximum variant spread {max_variant_spread:.1f} km s$^{{-1}}$", transform=ax.transAxes, ha="center", va="bottom", fontsize=8.0, color="#3A3A3A", clip_on=False)
    ax.set_xlabel("Radius [kpc]")
    ax.set_ylabel(r"Rotation velocity [km s$^{-1}$]")
    ax.grid(alpha=0.22, zorder=0)
    ax.set_xlim(left=0.0)
    ax.set_ylim(bottom=0.0)

handles, labels = axes[0].get_legend_handles_labels()
unique_legend = dict(zip(labels, handles))
fig.legend(unique_legend.values(), unique_legend.keys(), loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.03), fontsize=10)
fig.suptitle("Representative Galaxy Rotation Curves from Frozen Out-of-Fold Reconstructions", fontsize=16.5, fontweight="bold", y=1.07)
fig.text(0.5, -0.015, "Galaxies were selected deterministically at four percentiles of the Q12 galaxy-wise OOF error distribution.", ha="center", fontsize=9.5)
FIGURE_1_PNG = FIGURE_DIR / "figure_candidate_h_representative_galaxy_rotation_curves_compact_labels.png"
fig.savefig(FIGURE_1_PNG, dpi=300, bbox_inches="tight")
plt.close(fig)

CAPTION_1 = FIGURE_DIR / "figure_candidate_h_compact_labels_caption_en.txt"
CAPTION_1.write_text("Figure candidate H. Representative Q12 galaxy rotation curves from frozen out-of-fold reconstructions. Black points with error bars show the observed rotation curve, and the dashed gold line shows the baryonic contribution. The solid blue line is the median reconstruction across four pre-specified frozen kernel variants; the pale-blue band, bounded by fine blue lines, shows their actual minimum-to-maximum spread. Compact labels above each plotting area report the number of radial rows, the median-curve RMSE, and the maximum variation across the four frozen variants. The galaxies were selected deterministically by percentile of galaxy-wise Q12 OOF error.\n", encoding="utf-8")
MANIFEST_1 = FIGURE_DIR / "figure_candidate_h_compact_labels_manifest.json"
MANIFEST_1.write_text(json.dumps({"figure": FIGURE_1_PNG.name, "source_table": SOURCE_TABLE.name, "selection_table": SELECTION_TABLE.name, "caption": CAPTION_1.name, "n_kernel_variants": 4, "selection_labels": percentile_labels, "layout_note": "Compact metric labels are positioned above each plotting area and do not overlap observed data points.", "sha256": {FIGURE_1_PNG.name: sha256_file(FIGURE_1_PNG), CAPTION_1.name: sha256_file(CAPTION_1)}}, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("=== FIGURE 1 COMPLETE ===")
print("Figure:", FIGURE_1_PNG)
display(Image(filename=str(FIGURE_1_PNG)))




# ============================================================
# CELL 16 — FIGURE 2: OOF RESIDUAL PHASE MAP
# ============================================================

from matplotlib.colors import TwoSlopeNorm

data = pd.read_csv(EXPLORATION_SOURCE)
required_columns = {"Galaxy", "sample", "outer_fold", "kernel_variant", "log_g_bar", "observed_log_g_obs", "predicted_log_g_obs", "residual_dex"}
missing_columns = sorted(required_columns - set(data.columns))
if missing_columns:
    raise RuntimeError(f"Missing required source-data columns: {missing_columns}")
data = data[np.isfinite(data["log_g_bar"]) & np.isfinite(data["observed_log_g_obs"]) & np.isfinite(data["predicted_log_g_obs"]) & np.isfinite(data["residual_dex"])].copy()
if data.empty:
    raise RuntimeError("No finite Q12 OOF records are available.")
if not np.allclose(data["residual_dex"].to_numpy(), data["predicted_log_g_obs"].to_numpy() - data["observed_log_g_obs"].to_numpy(), rtol=0.0, atol=1e-12):
    raise RuntimeError("Figure 2 residual definition is inconsistent with the derived total-acceleration reconstruction.")

n_records, n_galaxies, n_variants = int(len(data)), int(data["Galaxy"].nunique()), int(data["kernel_variant"].nunique())
if n_variants != 4:
    raise RuntimeError(f"Expected four frozen kernel variants; found {n_variants}.")
n_unique_radial_rows = int(n_records // n_variants)
SOURCE_2 = FIGURE_DIR / "figure_2_oof_residual_phase_map_source_data.csv"
data[["Galaxy", "sample", "outer_fold", "kernel_variant", "log_g_bar", "observed_log_g_obs", "predicted_log_g_obs", "residual_dex"]].to_csv(SOURCE_2, index=False)

colour_limit = max(float(np.quantile(np.abs(data["residual_dex"].to_numpy(dtype=float)), 0.95)), 0.05)
fig, ax = plt.subplots(figsize=(9.2, 7.6))
hexbin = ax.hexbin(data["log_g_bar"], data["observed_log_g_obs"], C=data["residual_dex"], reduce_C_function=np.median, gridsize=36, mincnt=5, cmap="RdBu_r", norm=TwoSlopeNorm(vcenter=0.0, vmin=-colour_limit, vmax=colour_limit), linewidths=0.28, edgecolors="white", zorder=2)
axis_lower = float(min(data["log_g_bar"].min(), data["observed_log_g_obs"].min()) - 0.10)
axis_upper = float(max(data["log_g_bar"].max(), data["observed_log_g_obs"].max()) + 0.10)
ax.plot([axis_lower, axis_upper], [axis_lower, axis_upper], color="#202020", linewidth=1.35, linestyle="--", label=r"Baryon-only equality: $g_{\rm obs}=g_{\rm bar}$", zorder=3)
colorbar = fig.colorbar(hexbin, ax=ax, pad=0.02)
colorbar.set_label("Median OOF reconstruction residual [dex]\n" + r"$\log_{10}(g_{\rm obs,pred})-\log_{10}(g_{\rm obs})$", fontsize=10.5)
ax.set_xlim(axis_lower, axis_upper)
ax.set_ylim(axis_lower, axis_upper)
ax.set_xlabel(r"Baryonic acceleration, $\log_{10}(g_{\rm bar}/\mathrm{m\,s^{-2}})$", fontsize=11.5)
ax.set_ylabel(r"Observed acceleration, $\log_{10}(g_{\rm obs}/\mathrm{m\,s^{-2}})$", fontsize=11.5)
ax.set_title("Out-of-Fold Residual Structure Across the Acceleration Plane", fontsize=16, fontweight="bold", pad=14)
ax.grid(alpha=0.18, zorder=0)
ax.legend(loc="lower right", frameon=True, fontsize=9.5)
ax.text(0.035, 0.965, "\n".join(["Q12 sample", f"{n_galaxies} galaxies", f"{n_unique_radial_rows:,} unique radial rows", f"{n_variants} frozen kernel variants", "Blue: under-reconstruction", "Red: over-reconstruction"]), transform=ax.transAxes, ha="left", va="top", fontsize=9.6, bbox=dict(boxstyle="round,pad=0.42", facecolor="white", edgecolor="#777777", alpha=0.94), zorder=4)
fig.text(0.5, 0.015, "Each coloured hexagon reports the median certified OOF residual of at least five prediction records.", ha="center", fontsize=9.5)
FIGURE_2_PNG = FIGURE_DIR / "figure_2_oof_residual_phase_map.png"
fig.savefig(FIGURE_2_PNG, dpi=300, bbox_inches="tight")
plt.close(fig)

CAPTION_2 = FIGURE_DIR / "figure_2_caption_en.txt"
CAPTION_2.write_text("Figure 2. Out-of-fold residual structure across the acceleration plane in the Q12 sample. Each hexagon shows the median residual between the frozen OOF reconstruction and observed total acceleration for at least five prediction records. Blue regions indicate systematic under-reconstruction and red regions systematic over-reconstruction. The dashed line marks the baryon-only equality, g_obs = g_bar. The 11,220 prediction records represent 2,805 unique radial rows from 163 galaxies, evaluated under four pre-specified frozen kernel variants. The colour scale is symmetric about zero and clipped visually at the 95th percentile of absolute residual; no source records are removed.\n", encoding="utf-8")
MANIFEST_2 = FIGURE_DIR / "figure_2_manifest.json"
MANIFEST_2.write_text(json.dumps({"figure": FIGURE_2_PNG.name, "source_table": SOURCE_2.name, "caption": CAPTION_2.name, "source_provenance": EXPLORATION_SOURCE.relative_to(RUN_DIR).as_posix(), "sample": "Q12", "n_oof_prediction_records": n_records, "n_unique_radial_rows": n_unique_radial_rows, "n_galaxies": n_galaxies, "n_kernel_variants": n_variants, "hexbin_grid_size": 36, "hexbin_minimum_records": 5, "colour_scale": {"centre_dex": 0.0, "symmetric_limit_dex": colour_limit, "visual_clipping_percentile": 95}, "sha256": {FIGURE_2_PNG.name: sha256_file(FIGURE_2_PNG), SOURCE_2.name: sha256_file(SOURCE_2), CAPTION_2.name: sha256_file(CAPTION_2)}}, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("=== FIGURE 2 COMPLETE ===")
print(f"OOF prediction records: {n_records:,}")
print(f"Unique radial rows: {n_unique_radial_rows:,}")
print(f"Figure: {FIGURE_2_PNG}")
display(Image(filename=str(FIGURE_2_PNG)))


# SPARC FROZEN VALIDATION REPRODUCTION — CELL 4
# Writes the English validation certificate and checksums for this run.

from importlib.metadata import version
import platform

if "field_comparison" not in globals() or "gobs_comparison" not in globals():
    raise RuntimeError("Run CELL 2 and CELL 3 first.")

field_pass = bool(field_comparison["field_metrics_exact"].all())
gobs_pass = bool(gobs_comparison["gobs_metrics_exact"].all())
overall_pass = bool(field_pass and gobs_pass)

certificate_lines = [
    "# SPARC frozen validation reproduction certificate",
    "",
    "## Result",
    "",
    f"Overall validation status: {'PASS' if overall_pass else 'FAIL'}",
    f"Field-response metrics reproduced exactly: {field_pass}",
    f"Reconstructed observed-acceleration metrics reproduced exactly: {gobs_pass}",
    "",
    "## Scope",
    "",
    "This notebook re-executes the frozen SPARC kernel-variant nested-validation",
    "design from archived derived inputs, archived kernel features, frozen outer",
    "fold assignments, frozen model specifications, and frozen Ridge parameters.",
    "",
    "The official SPARC raw archive is not redistributed by this release.",
    "",
    "## Numerical criterion",
    "",
    "A metric is treated as exact when the absolute difference from the locked",
    "reference is below 1e-10. Observed residual differences of order 1e-16 are",
    "standard floating-point rounding effects.",
    "",
    "## Field-response comparison",
    "",
    field_comparison.to_markdown(index=False),
    "",
    "## Reconstructed observed-acceleration comparison",
    "",
    gobs_comparison.to_markdown(index=False),
    "",
    "## Execution environment",
    "",
    f"- Python: {platform.python_version()}",
    f"- numpy: {version('numpy')}",
    f"- pandas: {version('pandas')}",
    f"- scikit-learn: {version('scikit-learn')}",
]

certificate_path = RUN_DIR / "VALIDATION_CERTIFICATE.md"

certificate_path.write_text(
    "\n".join(certificate_lines) + "\n",
    encoding="utf-8",
)

summary_payload = {
    "validation_status": "PASS" if overall_pass else "FAIL",
    "field_metrics_exact": field_pass,
    "gobs_metrics_exact": gobs_pass,
    "metric_tolerance": 1e-10,
    "n_frozen_outer_fold_fits": int(len(exact_fold_check)),
    "n_reconstructed_oof_rows": int(len(exact_oof)),
    "samples": sorted(field_comparison["sample"].unique().tolist()),
    "kernel_variants": sorted(
        field_comparison["kernel_variant"].unique().tolist()
    ),
}

summary_path = RUN_DIR / "validation_summary.json"

summary_path.write_text(
    json.dumps(summary_payload, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

checksum_lines = [
    "# SHA-256 manifest for this frozen-validation reproduction run"
]

run_files = sorted(
    path for path in RUN_DIR.rglob("*")
    if path.is_file() and path.name != "SHA256SUMS.txt"
)

for path in run_files:
    relative_name = path.relative_to(RUN_DIR).as_posix()
    checksum_lines.append(f"{sha256_file(path)}  {relative_name}")

checksum_path = RUN_DIR / "SHA256SUMS.txt"

checksum_path.write_text(
    "\n".join(checksum_lines) + "\n",
    encoding="utf-8",
)

print("=== FROZEN VALIDATION CERTIFIED ===")
print("Overall status:", "PASS" if overall_pass else "FAIL")
print("Field metrics exact:", field_pass)
print("g_obs metrics exact:", gobs_pass)
print("Frozen outer-fold fits:", len(exact_fold_check))
print("Reconstructed OOF rows:", len(exact_oof))
print("Certificate:", certificate_path)
print("Summary:", summary_path)
print("Checksums:", checksum_path)

if not overall_pass:
    raise RuntimeError("Frozen validation did not pass.")



# ============================================================
# FINAL PORTABLE RELEASE ARCHIVE
# ============================================================

import shutil
archive_base = RUN_DIR.parent / f"{RUN_DIR.name}_with_figures"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR))
print("=== ONE-CELL MASTER COMPLETE ===")
print("Certified run directory:", RUN_DIR)
print("Portable release ZIP:", archive_path)
